# <h1 style='background:#C4F1E8; border:2; border-radius: 10px; font-size:250%; font-weight: bold; color:black'><center>Brain Stroke Prediction</center></h1> 
 
<img src = "https://www.news-medical.net/image.axd?picture=2021%2F11%2Fshutterstock_1488035675-1.jpg" width = 500 height = 400/>

A stroke is an interruption of the blood supply to any part of the brain. If blood flow was stopped for longer than a few seconds and the brain cannot get blood and oxygen, brain cells can die, and the abilities controlled by that area of the brain are lost.
In this Notebook we will use some features to see wether we will be able to predict the stoke or not? 
<a id='top'></a>
<div class="list-group" id="list-tab" role="tablist">
    
<h1 style='background:#C4F1E8; border:0; border-radius: 10px; color:black'><center> TABLE OF CONTENTS </center></h1>

### [**1. IMPORTING LIBRARIES AND LOADING DATA**](#title-one)

### [**2. DATA EXPLORATION**](#title-two)
  
### [**3. VIZUALIZATION**](#title-four) 
    
### [**4. DATA PREPROCESSING**](#title-five)

### [**5. MODEL BUILDING**](#title-six)
    
<a id="title-one"></a>
<h1 style='background:#C4F1E8; border:2; border-radius: 10px; color:black'><center>IMPORTING LIBRARIES AND LOADING DATA</center></h1>

In [ ]:
import numpy as np 
import pandas as pd 
import os
import matplotlib.pyplot as plt
import seaborn as sns
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
import warnings
from collections import Counter
warnings.filterwarnings('ignore')
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix,classification_report
from imblearn.under_sampling import RandomUnderSampler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split,GridSearchCV

In [ ]:
df = pd.read_csv('../input/full-filled-brain-stroke-dataset/full_data.csv')
df.head(10)

<a id="title-two"></a>
<h1 style='background:#C4F1E8; border:2; border-radius: 10px; color:black'><center>DATA EXPLORATION</center></h1>

In [ ]:
df.info()

**Turning to Numerical**

## Numerical columns info

In [ ]:
df.describe()

## Categorical columns info

In [ ]:
print(df['gender'].unique())
print(df['work_type'].unique())
print(df['Residence_type'].unique())
print(df['smoking_status'].unique())
print(df['ever_married'].unique())

<a id="title-four"></a>
<h1 style='background:#C4F1E8; border:2; border-radius: 10px; color:black'><center>VIZUALIZATION</center></h1>

*I tried to use plotly for vizualization as I needed to learn it more :) *

In [ ]:
import cufflinks as cf
cf.go_offline()
cf.set_config_file(offline=False, world_readable=True)  #  link pandas to plotly and add the iplot method

<h3 style='background:#F1C4EF; border:1; border-radius:10px; color:black'><center>The Proportion of Stroke among Gender</center></h3>

In [ ]:
gender = df.groupby(df['gender'])['stroke'].sum()
df_gender = pd.DataFrame({'labels': gender.index,
                   'values': gender.values
                  })
colors = ['lightpink', 'lightskyblue']
df_gender.iplot(kind='pie',labels='labels',values='values', title='The Proportion of Stroke among Gender', colors = colors)

<h3 style='background:#F1C4EF; border:1; border-radius:10px; color:black'><center>Work type of people who had stroke</center></h3>

In [ ]:
job = df.groupby(df['work_type'])['stroke'].sum()
df_job = pd.DataFrame({'labels': job.index,
                   'values': job.values
                  })
colors2= ['palegreen','paleturquoise','thistle','moccasin']
df_job.iplot(kind='pie',labels='labels',values='values', title='Work type of people who had stroke', colors = colors2, 
            pull=[0.1, 0.1, 0.1, 0.2])


<h3 style='background:#F1C4EF; border:1; border-radius:10px; color:black'><center>Smoking status of people who had stroke</center></h3>

In [ ]:
smoke = df.groupby(df['smoking_status'])['stroke'].sum()
df_smoke = pd.DataFrame({'labels': smoke.index,
                   'values': smoke.values
                  })
df_smoke.iplot(kind='pie',labels='labels',values='values', title='Smoking status of people who had stroke', colors = colors2, 
            pull=[0.02, 0.02, 0.1, 0.02])


<h3 style='background:#F1C4EF; border:1; border-radius:10px; color:black'><center>Residence area of people who had stroke</center></h3>

In [ ]:

Residence = df.groupby(df['Residence_type'])['stroke'].sum()
df_Residence = pd.DataFrame({'labels': Residence.index,
                   'values': Residence.values
                  })
df_Residence.iplot(kind='pie',labels='labels',values='values', title='Residence area of people who had stroke', colors = colors2, 
            pull=[0.02, 0.02],hole = 0.3)

<h3 style='background:#F1C4EF; border:1; border-radius:10px; color:black'><center>Marriage status of people who had stroke</center></h3>

In [ ]:
Married = df.groupby(df['ever_married'])['stroke'].sum()
df_Married = pd.DataFrame({'labels': Married.index,
                   'values': Married.values
                  })
df_Married.iplot(kind='pie',labels='labels',values='values', title='Marriage status of people who had stroke', colors = colors2, 
            pull=[0.02, 0.02],hole = 0.3)

<img src = "https://media1.giphy.com/media/jIIduCs5nkq2i2emoS/giphy.gif" width = 400 height = 300 class="center">



<h3 style='background:#F1C4EF; border:1; border-radius:10px; color:black'><center>Stroke age among gender</center></h3>

In [ ]:
stroke = df.loc[df['stroke']== 1].reset_index()

stroke["male_age"]=stroke[stroke["gender"]=="Male"]["age"]
stroke["female_age"]=stroke[stroke["gender"]=="Female"]["age"]
stroke[["male_age","female_age"]].iplot(kind="histogram", bins=20, theme="white", title="Stroke Ages",
         xTitle='Ages', yTitle='Count')

You are able to turn on and off the gender for better vizualisation

<a id="title-five"></a>
<h1 style='background:#C4F1E8; border:2; border-radius: 10px; color:black'><center>DATA PREPROCESSING</center></h1>

<h3 style='background:#F1C4EF; border:1; border-radius:10px; color:black'><center>Handling Categorical Columns</center></h3>

In [ ]:
df['ever_married'] = [ 0 if i !='Yes' else 1 for i in df['ever_married'] ]
df['gender'] = [0 if i != 'Female' else 1 for i in df['gender']]
df.head(5)

In [ ]:
df = pd.get_dummies(df, columns = ['work_type', 'Residence_type','smoking_status'])
df.sample(5)

In [ ]:
df.isnull().sum()

Let's see of the data is imbalanced or not?

In [ ]:
df['stroke'].value_counts()

<h3 style='background:#F1C4EF; border:1; border-radius:10px; color:black'><center>Target and Feature values / Train Test Split</center></h3>

In [ ]:
X = df.drop(['stroke'], axis = 1)
y = df['stroke']

In [ ]:
X_train, X_test, y_train , y_test = train_test_split(X,y, test_size = 0.33, random_state = 42)
X_train.shape, X_test.shape

<a id="title-six"></a>
<h1 style='background:#C4F1E8; border:2; border-radius: 10px; color:black'><center>MODEL BUILDING</center></h1>

<h3 style='background:#F1C4EF; border:1; border-radius:10px; color:black'><center>Decisiontree Classifier and Gini method</center></h3>

In [ ]:
clf_gini = DecisionTreeClassifier(criterion='gini', random_state=0,max_depth= 5)
clf_gini.fit(X_train, y_train)
y_pred_gini = clf_gini.predict(X_test)

<h3 style='background:#F1C4EF; border:1; border-radius:10px; color:black'><center>Model accuracy score</center></h3>

In [ ]:
print(confusion_matrix(y_test,y_pred_gini))
print('The classification report is:\n{:}'.format(classification_report(y_test,y_pred_gini)))

In [ ]:
df['stroke'].value_counts()

### The confusion  matrix and classification report shows that the model prediction is not going to be acceptable as the data is imbalance. The model predicted 81 Flase Negative which means that we did not predict 81 cases which are going to have stroke! Then it is a disaster!

<h3 style='background:#F1C4EF; border:1; border-radius:10px; color:black'><center>Under Sampling the data</center></h3>

#### To fix this issue, I decided to undersample the data:

In [ ]:
undersample = RandomUnderSampler(sampling_strategy='majority')
X_under, y_under = undersample.fit_resample(X, y)
print(sorted(Counter(y_under).items()))

#### Splitting the new train and test using our undersampled data

In [ ]:
X_train_rs, X_test_rs, y_train_rs , y_test_rs = train_test_split(X_under,y_under, test_size = 0.33, random_state = 43)
X_train_rs.shape, X_test_rs.shape

#### Fitting the model again:

In [ ]:
clf_gini.fit(X_train_rs,y_train_rs)

#### And prediction:

In [ ]:
y_pred_rs = clf_gini.predict(X_test_rs)
print(confusion_matrix(y_test_rs,y_pred_rs))
print('The classification report is:\n{:}'.format(classification_report(y_test_rs,y_pred_rs)))

#### Now the data is balanced but the model prediction si not good as enough. let's see what happened if we fit the random forest classifier:


<h3 style='background:#F1C4EF; border:1; border-radius:10px; color:black'><center>Random Forest Classifier</center></h3>

In [ ]:
rfc = RandomForestClassifier()
rfc.fit(X_train_rs,y_train_rs)

In [ ]:
y_pred_rfc = rfc.predict(X_test_rs)
print(confusion_matrix(y_test_rs,y_pred_rfc))

#### a little better, Now let's do the hyperparameter tuning

<h3 style='background:#F1C4EF; border:1; border-radius:10px; color:black'><center>Hyperparameter tuning</center></h3>

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
n_estimators = [90,100,115,130, 200]
max_depth = range(2,20,1)
min_samples_split = range(2,10,1)
min_samples_leaf = range(1,10,1)
criterion = ['gini']

param = dict(n_estimators = n_estimators, max_depth = max_depth,  
              min_samples_split = min_samples_split, 
             min_samples_leaf = min_samples_leaf,
             criterion = criterion)
rfc_random = RandomizedSearchCV(estimator = rfc, param_distributions = param, cv = 5, verbose=2, random_state=42)
rfc_random.fit(X_train_rs, y_train_rs)

In [ ]:
y_pred_rfc_random = rfc_random.predict(X_test_rs)
print(confusion_matrix(y_test_rs,y_pred_rfc_random))
print('The accuracy is: {:.4f}'.format(accuracy_score(y_test_rs,y_pred_rfc_random)))
print('The classification report is:\n{:}'.format(classification_report(y_test_rs,y_pred_rfc_random)))

#### This is what we get from the random forest at the best case.
#### I will continue this notebook to add more algorithms and Thanks to people who's comments make this notebook be more valuable. If you have any suggestion, let me know in the comments. 

##### Thanks to Feedbacks which leads to make this kernel more useful.